In [6]:
from matplotlib.colors import ListedColormap
import torch
import nrrd 
import matplotlib.pyplot as plt
import numpy as np
import torchvision.models as models 
from torch.utils.data import Dataset,DataLoader
from model.LeNet5 import LeNet
import numpy as np
import torch
import os
from PIL import Image, ImageOps
from torchvision import transforms
from pathlib import Path
from typing import Dict
import re

In [2]:
#Global variable
pattern=re.compile(r"^Segment(\d+)_Tags$")
Segment_signal_string="Segmentation.Status:inprogress"


In [ ]:
pattern=re.compile(r"^Segment(\d+)_Tags$")
Segment_signal_string="Segmentation.Status:inprogress"

def get_type_ACL(seg_header:Dict):
    seg_tags={k:v for k,v in seg_header.items() if ((m:=pattern.match(k)) and (0<=int(m.group(1))<=5))}
    for i in range(len(seg_tags)):
        if list(seg_tags.values())[i].startswith(Segment_signal_string):
            return i

def collect_data_and_seg(root_dir: str)->Dict:
    root_dir = Path(root_dir)
    result = {}
    for subfolder in root_dir.iterdir():
        if not subfolder.is_dir():
            continue
        # Find segmentation files
        seg_files = sorted(subfolder.glob("Segmentation*.seg.nrrd"))
        if not seg_files:
            continue
        # Find T2 files
        t2_files = sorted(
            p for p in subfolder.glob("*.nrrd")
            if "t2_tse_sag" in p.stem.lower()
        )
        if not t2_files:
            continue
        data,data_header=nrrd.read(t2_files[0])
        seg_data,seg_header=nrrd.read(seg_files[0])
        type_acl=get_type_ACL(seg_header)
        result[subfolder.name] = {
            "nrrd": data,
            "seg": seg_data,
            "type":type_acl
        }
    return result
root_dir=r"D:\personal_data\Duy Phan Ig\my_tool\Thuan dot 1"
data_dict=collect_data_and_seg(root_dir)
print(f"number of selected files: {len(data_dict)}")
for patient, data in data_dict.items():
    print(f"Patient {patient} with ACL type: {data['type']} ")



number of selected files: 20
Patient BUI NGUYEN BAO QUOC (18024987) with ACL type: 1 
Patient DANG NGOC THO (19069990) with ACL type: 0 
Patient HUYNH PHUC THANH (25918976) with ACL type: 0 
Patient HUYNH THANH PHONG (25919398) with ACL type: 0 
Patient KHUAT THI THU HANG (22940170) with ACL type: 4 
Patient MAI THI BICH LOC (25073019) with ACL type: 4 
Patient NGO HONG PHAN (25158695) with ACL type: 0 
Patient NGUYEN KHANH THANH (25916940) with ACL type: 4 
Patient NGUYEN LE HOANG MINH (25918812) with ACL type: 4 
Patient NGUYEN TAN TAI (25116149) with ACL type: 1 
Patient NGUYEN THI HONG HUE (25925602) with ACL type: 4 
Patient NGUYEN VAN SON (25141327) with ACL type: 1 
Patient NGUYEN XUAN BANG (23299803) with ACL type: 0 
Patient PHAM THI THAM (25060375) with ACL type: 1 
Patient PHAM TIEN DAT (25053224) with ACL type: 1 
Patient TAO QUANG CUONG (25136935) with ACL type: 1 
Patient TRAN THANH THOAI 22930629 with ACL type: 4 
Patient TRAN THANH TOAN 25924590 with ACL type: 3 
Patien

In [ ]:
class MRI_data(Dataset):
    def __init__(self,data_dict:Dict,img_trasform=None,label_transform=None,target_size=(320,320)):
        
    def __len__(self):
        return self.images.shape[2]
    def __getitem__(self, index): 
        
        return img,label 

IndentationError: expected an indented block after function definition on line 4 (1167496918.py, line 6)

In [55]:
img_transform=transforms.Compose([
    
    transforms.Resize((320,320)),
    transforms.ToTensor(),
])
label_transform=transforms.Compose([
    transforms.ToTensor(),
])

In [ ]:
dataset=MRI_data(root_nrrd_file_dir=r'D:\Project\ACL\my_tool\Thuan dot 1\KHUAT THI THU HANG (22940170)\8 t2_tse_sag.nrrd',
                 root_nrrd_seg_dir=r'D:\Project\ACL\my_tool\Thuan dot 1\KHUAT THI THU HANG (22940170)\Segmentation.seg.nrrd',
                 img_trasform=img_transform,
                 label_transform=label_transform)
print(f"Dataset length: {len(dataset)}")
print(f"Sample image shape: {dataset[0][0].shape}, Sample label: {dataset[0][1]}")
test_loader=DataLoader(dataset=dataset,batch_size=4,shuffle=True,)
for images,labels in test_loader:
    print(f"Batch image shape: {images.shape}, Batch labels: {labels}")
    break

Dataset length: 25


TypeError: Unexpected type <class 'numpy.ndarray'>

In [45]:
para = {
        'input_image_size': (320, 320),
        'input_channel': 1,
        'number_of_conv_layer':2,
        'number_of_fc_layer':2,
        'num_classes':2,
        'conv_channels': [6, 16],
        'fc_feature':[84,2]
    }
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=LeNet(para)
model.load_state_dict(torch.load('model.pth',weights_only=False))
model.to(device)
model.eval()

LeNet(
  (Conv_layer): ModuleList(
    (0): Sequential(
      (0): Conv2d(1, 6, kernel_size=(7, 7), stride=(1, 1), padding=(2, 2))
      (1): BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(6, 16, kernel_size=(7, 7), stride=(1, 1), padding=(2, 2))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (FC_layer): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=97344, out_features=84, bias=True)
      (1): ReLU()
      (2): Linear(in_features=84, out_features=2, bias=True)
      (3): Softmax(dim=1)
    )
  )
)